In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
# Consolidated Data Pipeline for LLM Training
# Based on Chapter 2 of "Build a Large Language Model From Scratch"

import os
import requests
import torch
import tiktoken
from torch.utils.data import Dataset, DataLoader

# ==========================================
# STEP 1: DEFINE THE DATASET
# ==========================================
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text using BytePair Encoding (BPE)
        # allowed_special is used to keep special tags like <|endoftext|> intact
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the text into overlapping sequences
        for i in range(0, len(token_ids) - max_length, stride):
            # Input chunk (what the model sees)
            input_chunk = token_ids[i : i + max_length]
            # Target chunk (shifted by 1 - what the model needs to predict)
            target_chunk = token_ids[i + 1 : i + max_length + 1]
            
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

# ==========================================
# STEP 2: DEFINE THE DATALOADER
# ==========================================
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True, num_workers=0):
    
    # Initialize the GPT-2 tokenizer (tiktoken is OpenAI's fast tokenizer)
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader (groups data into batches for the GPU)
    dataloader = DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=shuffle, 
        drop_last=drop_last, 
        num_workers=num_workers
    )

    return dataloader

# ==========================================
# STEP 3: MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    
    # 1. Download the raw text data (The Verdict short story)
    file_path = "the-verdict.txt"
    if not os.path.exists(file_path):
        url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        with open(file_path, "wb") as f:
            f.write(response.content)
            
    with open(file_path, "r", encoding="utf-8") as f:
        raw_text = f.read()

    # 2. Hyperparameters for the embedding layers (matching GPT-2 specifics)
    vocab_size = 50257      # Total number of unique tokens in GPT-2's vocabulary
    output_dim = 256        # Size of the embedding vectors
    context_length = 1024   # Maximum sequence length the model can handle
    batch_size = 8          # Number of text chunks processed at once
    max_length = 4          # Sequence length for this specific training run

    # 3. Create the Dataloader
    dataloader = create_dataloader_v1(
        txt=raw_text,
        batch_size=batch_size,
        max_length=max_length,
        stride=max_length, # Stride = max_length means no overlap between chunks
        shuffle=False
    )

    # 4. Initialize Embedding Layers
    # Maps Token IDs -> Vectors
    token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
    # Maps Position -> Vectors
    pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

    # 5. Process a single batch to demonstrate the pipeline
    for batch in dataloader:
        x, y = batch # x is input, y is target
        
        # A) Token Embeddings: Convert word IDs into numbers
        token_embeddings = token_embedding_layer(x)
        
        # B) Positional Embeddings: Create vectors representing the position (0, 1, 2, 3)
        pos_embeddings = pos_embedding_layer(torch.arange(max_length))
        
        # C) Final Input Embeddings: Add them together
        input_embeddings = token_embeddings + pos_embeddings
        
        print(f"Input Shape (x): {x.shape} -> (Batch Size, Sequence Length)")
        print(f"Target Shape (y): {y.shape} -> (Batch Size, Sequence Length)")
        print(f"Final Input Embeddings Shape: {input_embeddings.shape} -> (Batch Size, Sequence Length, Embedding Dimension)")
        break # Only process the first batch for demonstration

Input Shape (x): torch.Size([8, 4]) -> (Batch Size, Sequence Length)
Target Shape (y): torch.Size([8, 4]) -> (Batch Size, Sequence Length)
Final Input Embeddings Shape: torch.Size([8, 4, 256]) -> (Batch Size, Sequence Length, Embedding Dimension)


In [2]:
from importlib.metadata import version
import os
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

# 1. Print system environment metrics
print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))
print("CUDA Available:", torch.cuda.is_available())

# 2. Define the exact sliding-window Dataset for Autoregressive Next-Token Prediction
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text block, keeping document boundary tokens intact
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the token streams into sequence fragments
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1] # Shifted by 1 position
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

# 3. Create the automated DataLoader abstraction wrapper
def create_dataloader_v1(txt, batch_size, max_length, stride,
                         shuffle=True, drop_last=True, num_workers=0):
    # Initialize the high-performance Rust-backed BPE tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Instantiate the custom dataset with token sequences
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Initialize the multi-threaded PyTorch DataLoader pipeline
    dataloader = DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=shuffle, 
        drop_last=drop_last, 
        num_workers=num_workers
    )

    return dataloader

# 4. Safely load the raw dataset text using your designated Kaggle environment path
file_path = "/kaggle/input/datasets/yashwantguttulla/the-verdict/the-verdict.txt"

if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        raw_text = f.read()
    print(f"\nSuccessfully loaded text asset. Total characters: {len(raw_text)}")
else:
    raise FileNotFoundError(f"Could not locate the file at {file_path}. Please verify your Kaggle input mount settings.")

# 5. Define Model Hyperparameters (Matching standard GPT-2 configuration rules)
vocab_size = 50257       # Full BPE tokenizer vocabulary size
output_dim = 256         # Dimensionality of the dense latent space vectors
context_length = 1024    # Maximum permissible input context sequence path length

# 6. Instantiate the Parallel Trainable Embedding Vectors
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

# 7. Build the DataLoader with micro-batch settings for initial evaluation
batch_size = 8
max_length = 4  # Short context window for rapid matrix shape analysis
stride = max_length

dataloader = create_dataloader_v1(
    raw_text,
    batch_size=batch_size,
    max_length=max_length,
    stride=stride,
    shuffle=True,
    drop_last=True
)

# 8. Pull a singular operational batch to test structural vector addition
for batch in dataloader:
    x, y = batch  # x: Inputs tensor, y: Expected Targets tensor (shifted right by 1)

    # Transform token IDs directly into structural vector arrays
    token_embeddings = token_embedding_layer(x)
    
    # Generate relative coordinates from 0 to max_length for positional awareness
    position_indices = torch.arange(max_length)
    pos_embeddings = pos_embedding_layer(position_indices)

    # Linear vector addition injecting semantic context along with token ordering rules
    input_embeddings = token_embeddings + pos_embeddings
    break

# 9. Validate dimensions of the resulting tensor
print("\n--- Structural Dimensions Report ---")
print("Input Batch shape (x)            :", x.shape)
print("Target Batch shape (y)           :", y.shape)
print("Token Embeddings component shape :", token_embeddings.shape)
print("Positional Embeddings shape      :", pos_embeddings.shape)
print("Final Combined Input Tensor shape:", input_embeddings.shape)

torch version: 2.10.0+cu128
tiktoken version: 0.12.0
CUDA Available: True

Successfully loaded text asset. Total characters: 20479

--- Structural Dimensions Report ---
Input Batch shape (x)            : torch.Size([8, 4])
Target Batch shape (y)           : torch.Size([8, 4])
Token Embeddings component shape : torch.Size([8, 4, 256])
Positional Embeddings shape      : torch.Size([4, 256])
Final Combined Input Tensor shape: torch.Size([8, 4, 256])


In [3]:
import os
import json
import regex as re
import requests
from tqdm import tqdm
from functools import lru_cache
import torch
from torch.utils.data import Dataset, DataLoader

# ==========================================
# STEP 1: BPE Helper Functions
# ==========================================
@lru_cache()
def bytes_to_unicode():
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs = bs[:]
    n = 0
    for b in range(2**8):
        if b not in bs:
            bs.append(b)
            cs.append(2**8 + n)
            n += 1
    cs = [chr(n) for n in cs]
    return dict(zip(bs, cs))

def get_pairs(word):
    pairs = set()
    prev_char = word[0]
    for char in word[1:]:
        pairs.add((prev_char, char))
        prev_char = char
    return pairs

# ==========================================
# STEP 2: The Original OpenAI BPE Encoder
# ==========================================
class Encoder:
    def __init__(self, encoder, bpe_merges, errors="replace"):
        self.encoder = encoder
        self.decoder = {v: k for k, v in self.encoder.items()}
        self.errors = errors
        self.byte_encoder = bytes_to_unicode()
        self.byte_decoder = {v: k for k, v in self.byte_encoder.items()}
        self.bpe_ranks = dict(zip(bpe_merges, range(len(bpe_merges))))
        self.cache = {}
        self.pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

    def bpe(self, token):
        if token in self.cache:
            return self.cache[token]
        word = tuple(token)
        pairs = get_pairs(word)

        if not pairs:
            return token

        while True:
            bigram = min(pairs, key=lambda pair: self.bpe_ranks.get(pair, float("inf")))
            if bigram not in self.bpe_ranks:
                break
            first, second = bigram
            new_word = []
            i = 0
            while i < len(word):
                try:
                    j = word.index(first, i)
                    new_word.extend(word[i:j])
                    i = j
                except ValueError:
                    new_word.extend(word[i:])
                    break

                if word[i] == first and i < len(word) - 1 and word[i + 1] == second:
                    new_word.append(first + second)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_word = tuple(new_word)
            word = new_word
            if len(word) == 1:
                break
            else:
                pairs = get_pairs(word)
        word = " ".join(word)
        self.cache[token] = word
        return word

    def encode(self, text):
        bpe_tokens = []
        for token in re.findall(self.pat, text):
            token = "".join(self.byte_encoder[b] for b in token.encode("utf-8"))
            bpe_tokens.extend(self.encoder[bpe_token] for bpe_token in self.bpe(token).split(" "))
        return bpe_tokens

    def decode(self, tokens):
        text = "".join([self.decoder[token] for token in tokens])
        text = bytearray([self.byte_decoder[c] for c in text]).decode("utf-8", errors=self.errors)
        return text

# ==========================================
# STEP 3: Vocab Downloader & Loader
# ==========================================
def download_vocab(target_dir="/kaggle/working/gpt2_model"):
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)

    for filename in ["encoder.json", "vocab.bpe"]:
        file_path = os.path.join(target_dir, filename)
        if not os.path.exists(file_path): # Only download if it doesn't exist
            r = requests.get("https://openaipublic.blob.core.windows.net/gpt-2/models/117M/" + filename, stream=True)
            with open(file_path, "wb") as f:
                file_size = int(r.headers["content-length"])
                chunk_size = 1000
                with tqdm(ncols=100, desc="Fetching " + filename, total=file_size, unit_scale=True) as pbar:
                    for chunk in r.iter_content(chunk_size=chunk_size):
                        f.write(chunk)
                        pbar.update(chunk_size)

def get_encoder(models_dir):
    with open(os.path.join(models_dir, "encoder.json"), "r") as f:
        encoder = json.load(f)
    with open(os.path.join(models_dir, "vocab.bpe"), "r", encoding="utf-8") as f:
        bpe_data = f.read()
    bpe_merges = [tuple(merge_str.split()) for merge_str in bpe_data.split("\n")[1:-1]]
    return Encoder(encoder=encoder, bpe_merges=bpe_merges)

# ==========================================
# STEP 4: The Dataset and DataLoader
# ==========================================
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Encode the text using our custom BPE tokenizer
        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

# ==========================================
# STEP 5: Execution Pipeline
# ==========================================
# 5.1 Download and Initialize the Tokenizer
vocab_directory = "/kaggle/working/gpt2_model"
print("Downloading/Verifying GPT-2 Vocab files...")
download_vocab(target_dir=vocab_directory)
custom_tokenizer = get_encoder(models_dir=vocab_directory)

# 5.2 Load the text dataset
file_path = "/kaggle/input/datasets/yashwantguttulla/the-verdict/the-verdict.txt"
if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        raw_text = f.read()
    print(f"Successfully loaded text asset. Total characters: {len(raw_text)}")
else:
    raise FileNotFoundError(f"Could not locate the file at {file_path}.")

# 5.3 Configure Hyperparameters
vocab_size = 50257
output_dim = 256
context_length = 1024
batch_size = 8
max_length = 4
stride = max_length

# 5.4 Create Dataset and Dataloader
dataset = GPTDatasetV1(raw_text, custom_tokenizer, max_length, stride)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

# 5.5 Create Embeddings Layers
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

# 5.6 Process a single batch
print("\nProcessing Batch...")
for batch in dataloader:
    x, y = batch

    token_embeddings = token_embedding_layer(x)
    position_indices = torch.arange(max_length)
    pos_embeddings = pos_embedding_layer(position_indices)

    # Combine Token and Positional Embeddings
    input_embeddings = token_embeddings + pos_embeddings
    break

print("\n--- Structural Dimensions Report ---")
print("Input Batch shape (x)            :", x.shape)
print("Target Batch shape (y)           :", y.shape)
print("Final Combined Input Tensor shape:", input_embeddings.shape)

Downloading/Verifying GPT-2 Vocab files...


Fetching encoder.json: 1.04Mit [00:00, 1.21Mit/s]                                                   
Fetching vocab.bpe: 457kit [00:00, 619kit/s]                                                        


Successfully loaded text asset. Total characters: 20479

Processing Batch...

--- Structural Dimensions Report ---
Input Batch shape (x)            : torch.Size([8, 4])
Target Batch shape (y)           : torch.Size([8, 4])
Final Combined Input Tensor shape: torch.Size([8, 4, 256])


In [4]:
import os
import timeit
import tiktoken
from transformers import GPT2Tokenizer, GPT2TokenizerFast

# ==========================================
# STEP 1: Load the Dataset
# ==========================================
file_path = "/kaggle/input/datasets/yashwantguttulla/the-verdict/the-verdict.txt"

if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        raw_text = f.read()
    print(f"Loaded '{file_path.split('/')[-1]}' - Total characters: {len(raw_text)}\n")
else:
    raise FileNotFoundError(f"Could not locate {file_path}")

# ==========================================
# STEP 2: Initialize Tokenizers
# ==========================================
print("Initializing Tokenizers...")
# 1. OpenAI's official open-source tokenizer (written in Rust)
tik_tokenizer = tiktoken.get_encoding("gpt2")

# 2. Hugging Face standard GPT-2 tokenizer (written in Python)
hf_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# 3. Hugging Face FAST GPT-2 tokenizer (written in Rust)
hf_tokenizer_fast = GPT2TokenizerFast.from_pretrained("gpt2")

# ==========================================
# STEP 3: Validate Outputs (Sanity Check)
# ==========================================
# Ensure all tokenizers produce the exact same integer sequence
test_string = "Hello, world. Is this-- a test?"
print("\n--- Output Validation ---")
print("Tiktoken: ", tik_tokenizer.encode(test_string, allowed_special={"<|endoftext|>"}))
print("HF Fast : ", hf_tokenizer_fast(test_string)["input_ids"])
print("HF Std  : ", hf_tokenizer(test_string)["input_ids"])

# ==========================================
# STEP 4: Performance Benchmarking
# ==========================================
print("\n--- Speed Benchmarks ---")
print("Running tests... (this may take a few seconds)\n")

# Note: We use lambda functions here to pass the exact execution to the timeit module

# Test 1: Tiktoken
tik_time = timeit.timeit(lambda: tik_tokenizer.encode(raw_text), number=100)
print(f"1. Tiktoken (Rust)              : {tik_time:.4f} seconds (for 100 iterations)")

# Test 2: Hugging Face Fast
# We set truncation=True because HF throws a warning if sequence length > 1024
hf_fast_time = timeit.timeit(
    lambda: hf_tokenizer_fast(raw_text, truncation=True, max_length=10000)["input_ids"], 
    number=100
)
print(f"2. Hugging Face FAST (Rust)     : {hf_fast_time:.4f} seconds (for 100 iterations)")

# Test 3: Hugging Face Standard
hf_std_time = timeit.timeit(
    lambda: hf_tokenizer(raw_text, truncation=True, max_length=10000)["input_ids"], 
    number=100
)
print(f"3. Hugging Face Standard (Python): {hf_std_time:.4f} seconds (for 100 iterations)")

Loaded 'the-verdict.txt' - Total characters: 20479

Initializing Tokenizers...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


--- Output Validation ---
Tiktoken:  [15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]
HF Fast :  [15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]
HF Std  :  [15496, 11, 995, 13, 1148, 428, 438, 257, 1332, 30]

--- Speed Benchmarks ---
Running tests... (this may take a few seconds)

1. Tiktoken (Rust)              : 0.2179 seconds (for 100 iterations)
2. Hugging Face FAST (Rust)     : 0.9821 seconds (for 100 iterations)
3. Hugging Face Standard (Python): 0.9961 seconds (for 100 iterations)


In [5]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

# ==========================================
# EXERCISE 2.1: Tokenizer Fallback Behavior
# ==========================================
print("--- EXERCISE 2.1: BPE Subword Fallback ---")
tokenizer = tiktoken.get_encoding("gpt2")

# We give it a completely made-up word to see how BPE breaks it down
nonsense_word = "Akwirw ier"
integers = tokenizer.encode(nonsense_word)
print(f"Encoded array for '{nonsense_word}': {integers}")

# Decode piece by piece to see the subwords
for i in integers:
    print(f"ID {i} -> '{tokenizer.decode([i])}'")


# ==========================================
# EXERCISE 2.2: Dataloader Stride adjustments
# ==========================================
print("\n--- EXERCISE 2.2: Dataloader Windows ---")

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

def test_dataloader(txt, batch_size=4, max_length=256, stride=128):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    return DataLoader(dataset, batch_size=batch_size)

# Load the text 
file_path = "/kaggle/input/datasets/yashwantguttulla/the-verdict/the-verdict.txt"
with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

# Scenario A: max_length=2, stride=2 (No overlap)
loader_A = test_dataloader(raw_text, batch_size=4, max_length=2, stride=2)
for x, y in loader_A:
    print("\nScenario A Input Tensor (max_len=2, stride=2):")
    print(x)
    break

# Scenario B: max_length=8, stride=2 (Heavy overlap)
loader_B = test_dataloader(raw_text, batch_size=4, max_length=8, stride=2)
for x, y in loader_B:
    print("\nScenario B Input Tensor (max_len=8, stride=2):")
    print(x)
    break

--- EXERCISE 2.1: BPE Subword Fallback ---
Encoded array for 'Akwirw ier': [33901, 86, 343, 86, 220, 959]
ID 33901 -> 'Ak'
ID 86 -> 'w'
ID 343 -> 'ir'
ID 86 -> 'w'
ID 220 -> ' '
ID 959 -> 'ier'

--- EXERCISE 2.2: Dataloader Windows ---

Scenario A Input Tensor (max_len=2, stride=2):
tensor([[  40,  367],
        [2885, 1464],
        [1807, 3619],
        [ 402,  271]])

Scenario B Input Tensor (max_len=8, stride=2):
tensor([[   40,   367,  2885,  1464,  1807,  3619,   402,   271],
        [ 2885,  1464,  1807,  3619,   402,   271, 10899,  2138],
        [ 1807,  3619,   402,   271, 10899,  2138,   257,  7026],
        [  402,   271, 10899,  2138,   257,  7026, 15632,   438]])


In [6]:
import os
import re
import torch

# ==========================================
# STEP 1: Load Raw Text
# ==========================================
file_path = "/kaggle/input/datasets/yashwantguttulla/the-verdict/the-verdict.txt"

if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        raw_text = f.read()
else:
    raise FileNotFoundError(f"Could not locate {file_path}")

print("Total number of characters:", len(raw_text))
print("Sample:", raw_text[:99])

# ==========================================
# STEP 2: Basic Text Splitting (Regex)
# ==========================================
# We split on spaces, but also preserve punctuation as separate tokens
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
# Remove empty strings and pure whitespace
preprocessed = [item.strip() for item in preprocessed if item.strip()]

print("\n--- Regex Split Sample ---")
print(preprocessed[:30])
print("Total tokens in text:", len(preprocessed))

# ==========================================
# STEP 3: Building a Vocabulary
# ==========================================
# Create a sorted list of unique words/punctuation
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print("\nUnique Vocabulary Size:", vocab_size)

# Map string tokens to integer IDs
vocab = {token: integer for integer, token in enumerate(all_words)}

# ==========================================
# STEP 4: SimpleTokenizerV1 (The Naive Approach)
# ==========================================
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        # Maps string to int. Will CRASH if a word isn't in the vocab!
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Clean up spaces before punctuation
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

tokenizer_v1 = SimpleTokenizerV1(vocab)
test_sentence = '"It\'s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'
ids = tokenizer_v1.encode(test_sentence)
print("\nV1 Encoded IDs:", ids)
print("V1 Decoded Text:", tokenizer_v1.decode(ids))

# ==========================================
# STEP 5: SimpleTokenizerV2 (Handling Unknowns)
# ==========================================
# V1 crashes if it sees a new word. Let's add special tokens to fix that.
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab_v2 = {token: integer for integer, token in enumerate(all_tokens)}

class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        # If the word isn't in vocab, replace it with the <|unk|> token
        preprocessed = [
            item if item in self.str_to_int else "<|unk|>" for item in preprocessed
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

tokenizer_v2 = SimpleTokenizerV2(vocab_v2)
text1 = "Hello, do you like tea?" # 'Hello' and 'tea' are NOT in our book's vocab
text2 = "In the sunlit terraces of the palace."
combined_text = " <|endoftext|> ".join((text1, text2))

print("\n--- V2 Unknown Handling ---")
print("Original Text:", combined_text)
v2_ids = tokenizer_v2.encode(combined_text)
print("V2 Encoded IDs:", v2_ids)
print("V2 Decoded Output:", tokenizer_v2.decode(v2_ids))

# ==========================================
# STEP 6: Toy Embedding Example
# ==========================================
# How do these IDs become Neural Network inputs?
print("\n--- Embedding Layer Mechanics ---")
dummy_input_ids = torch.tensor([2, 3, 5, 1])
dummy_vocab_size = 6
dummy_output_dim = 3

torch.manual_seed(123)
# Creates a 6x3 matrix of random weights
embedding_layer = torch.nn.Embedding(dummy_vocab_size, dummy_output_dim)

print("Embedding Weight Matrix (6x3):\n", embedding_layer.weight)
print("\nVector for ID 3:\n", embedding_layer(torch.tensor([3])))
print("\nVectors for all dummy IDs [2, 3, 5, 1]:\n", embedding_layer(dummy_input_ids))

Total number of characters: 20479
Sample: I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 

--- Regex Split Sample ---
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']
Total tokens in text: 4690

Unique Vocabulary Size: 1130

V1 Encoded IDs: [1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
V1 Decoded Text: " It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.

--- V2 Unknown Handling ---
Original Text: Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.
V2 Encoded IDs: [1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]
V2 Decoded Output: <|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.

--- Embedding Layer 

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader

# 1. Create a dummy dataset of pure sequential numbers
with open("number-data.txt", "w", encoding="utf-8") as f:
    for number in range(1001):
        f.write(f"{number} ")

# 2. A modified Dataset that skips tiktoken and just parses integers
class NumberDataset(Dataset):
    def __init__(self, txt, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Parse the raw integers directly from the text file
        token_ids = [int(i) for i in txt.strip().split()]

        # The core sliding window logic
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

# Load the numbers
with open("number-data.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# 3. Test with a stride that matches max_length (No overlap)
dataset = NumberDataset(raw_text, max_length=4, stride=4)
dataloader = DataLoader(dataset, batch_size=2, shuffle=False)

print("--- Intuition Dataloader Output ---")
for inputs, targets in dataloader:
    print("Inputs (x):\n", inputs)
    print("Targets (y):\n", targets)
    break  # Just show the first batch

--- Intuition Dataloader Output ---
Inputs (x):
 tensor([[0, 1, 2, 3],
        [4, 5, 6, 7]])
Targets (y):
 tensor([[1, 2, 3, 4],
        [5, 6, 7, 8]])


In [2]:
import torch

# Suppose we have 3 token IDs from a vocabulary of 4 possible words
idx = torch.tensor([2, 3, 1])
num_vocab = 4
out_dim = 5

# --- Method A: The nn.Embedding Layer (The Shortcut) ---
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(num_vocab, out_dim)
embedded_output = embedding_layer(idx)

# --- Method B: One-Hot Encoding + Linear Matrix Multiplication ---
# 1. Convert to one-hot vectors
onehot = torch.nn.functional.one_hot(idx, num_classes=num_vocab)

# 2. Create a linear layer WITHOUT biases
torch.manual_seed(123)
linear_layer = torch.nn.Linear(num_vocab, out_dim, bias=False)

# 3. CRITICAL STEP: Copy the exact weights from the embedding layer 
# (Transposed to match matrix multiplication rules)
linear_layer.weight = torch.nn.Parameter(embedding_layer.weight.T)

# 4. Multiply the one-hot vectors by the weights
linear_output = linear_layer(onehot.float())

print("--- Mathematical Proof ---")
print("Output via nn.Embedding:\n", embedded_output)
print("\nOutput via One-Hot + nn.Linear:\n", linear_output)
# You will see the outputs are exactly the same!

--- Mathematical Proof ---
Output via nn.Embedding:
 tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

Output via One-Hot + nn.Linear:
 tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]], grad_fn=<MmBackward0>)


In [3]:
import re
from collections import Counter

class TinyBPETrainer:
    def __init__(self):
        self.vocab = {}
        self.bpe_merges = {}

    def train(self, text, target_vocab_size):
        # 1. Initialize vocabulary with individual characters
        unique_chars = sorted(list(set(text)))
        self.vocab = {i: char for i, char in enumerate(unique_chars)}
        inverse_vocab = {char: i for i, char in self.vocab.items()}
        
        # Convert text into a list of starting character IDs
        token_ids = [inverse_vocab[char] for char in text]
        
        # 2. Iteratively find and merge the most common pairs
        for new_id in range(len(self.vocab), target_vocab_size):
            if len(token_ids) < 2:
                break
                
            # Find the most frequent adjacent pair of IDs
            pairs = Counter(zip(token_ids, token_ids[1:]))
            if not pairs: break
            most_common_pair = max(pairs.items(), key=lambda x: x[1])[0]
            
            # Record the merge rule
            self.bpe_merges[most_common_pair] = new_id
            
            # Add the new merged string to our vocabulary
            merged_string = self.vocab[most_common_pair[0]] + self.vocab[most_common_pair[1]]
            self.vocab[new_id] = merged_string
            
            # 3. Replace all occurrences in the token list with the new ID
            new_token_ids = []
            i = 0
            while i < len(token_ids):
                if i < len(token_ids) - 1 and (token_ids[i], token_ids[i+1]) == most_common_pair:
                    new_token_ids.append(new_id)
                    i += 2 # Skip the merged token
                else:
                    new_token_ids.append(token_ids[i])
                    i += 1
            token_ids = new_token_ids

        return token_ids

# --- Test the Custom Trainer ---
trainer = TinyBPETrainer()
text_to_learn = "the cat in the hat"

# The text has 9 unique characters. Let's train it to a vocab size of 13.
# It should figure out that 't' + 'h' and 'h' + 'e' are very common.
final_ids = trainer.train(text_to_learn, target_vocab_size=13)

print("--- Custom Trained Vocabulary ---")
for id_num, string in trainer.vocab.items():
    print(f"ID {id_num}: '{string}'")

print("\nFinal compressed sequence of IDs:", final_ids)

--- Custom Trained Vocabulary ---
ID 0: ' '
ID 1: 'a'
ID 2: 'c'
ID 3: 'e'
ID 4: 'h'
ID 5: 'i'
ID 6: 'n'
ID 7: 't'
ID 8: 'th'
ID 9: 'the'
ID 10: 'the '
ID 11: 'at'
ID 12: 'the c'

Final compressed sequence of IDs: [12, 11, 0, 5, 6, 0, 10, 4, 11]


In [4]:
import torch
import torch.nn.functional as F

# =============================================================================
# TRUTH 1: THE DATABASE ANALOGY (Q, K, V)
# -----------------------------------------------------------------------------
# Attention works exactly like a database search.
# Query (Q): "What am I looking for?" (e.g., The word 'bank' looking for context)
# Key (K): "What do I contain?" (e.g., The word 'river' broadcasting its identity)
# Value (V): "What is my actual underlying meaning?"
# =============================================================================

# Let's simulate a tiny batch: 1 sequence, 3 tokens, 4 dimensions each
# Imagine the tokens are: "The", "river", "bank"
batch_size, context_length, embed_dim = 1, 3, 4
inputs = torch.randn(batch_size, context_length, embed_dim)

# We create three separate Linear layers to generate our Q, K, and V vectors
W_query = torch.nn.Linear(embed_dim, embed_dim, bias=False)
W_key   = torch.nn.Linear(embed_dim, embed_dim, bias=False)
W_value = torch.nn.Linear(embed_dim, embed_dim, bias=False)

queries = W_query(inputs) # Shape: [1, 3, 4]
keys    = W_key(inputs)   # Shape: [1, 3, 4]
values  = W_value(inputs) # Shape: [1, 3, 4]


# =============================================================================
# TRUTH 2: THE ATTENTION SCORES (Dot Product)
# -----------------------------------------------------------------------------
# To figure out how much the word "bank" should care about the word "river", 
# we multiply the Query of "bank" by the Key of "river". 
# A higher dot product means a stronger relationship.
# =============================================================================

# We transpose the Key matrix so we can multiply them: [1, 3, 4] @ [1, 4, 3] -> [1, 3, 3]
# The resulting 3x3 matrix tells us the relationship score between EVERY token.
attention_scores = queries @ keys.transpose(-2, -1) 

# We scale it down by the square root of the dimensions to keep the math stable
attention_scores = attention_scores / (embed_dim ** 0.5)


# =============================================================================
# TRUTH 3: CAUSAL MASKING (The "No Cheating" Rule)
# -----------------------------------------------------------------------------
# Because we are predicting the future (next token prediction), the word "The" 
# CANNOT be allowed to look at "river" or "bank" to figure out its context.
# We must mathematically block out the future by setting those scores to -Infinity.
# =============================================================================

# Create a triangle matrix of zeros and ones
mask = torch.tril(torch.ones(context_length, context_length))
# Example Mask:
# [1, 0, 0] -> Token 1 can only see Token 1
# [1, 1, 0] -> Token 2 can see Token 1 and 2
# [1, 1, 1] -> Token 3 can see Token 1, 2, and 3

# Replace the '0's (the future) with negative infinity
masked_scores = attention_scores.masked_fill(mask == 0, float('-inf'))


# =============================================================================
# TRUTH 4: SOFTMAX & THE CONTEXT VECTOR
# -----------------------------------------------------------------------------
# We convert the raw scores into percentages (weights) that sum to 100% (1.0).
# Finally, we multiply those weights by the Values to get our contextualized output.
# =============================================================================

# Softmax turns -inf into exactly 0.0 (perfectly erasing the future tokens)
attention_weights = F.softmax(masked_scores, dim=-1)

# Multiply the weights by the actual Values: [1, 3, 3] @ [1, 3, 4] -> [1, 3, 4]
context_vectors = attention_weights @ values

print("--- Self-Attention Pipeline ---")
print("1. Raw Inputs shape      :", inputs.shape)
print("2. Attention Mask        :\n", mask)
print("3. Attention Weights     :\n", attention_weights[0].detach().numpy().round(2))
print("4. Context Vectors shape :", context_vectors.shape)
# Notice the input shape [1, 3, 4] exactly matches the output shape [1, 3, 4].
# The difference? The output vectors are now richly infused with context.

--- Self-Attention Pipeline ---
1. Raw Inputs shape      : torch.Size([1, 3, 4])
2. Attention Mask        :
 tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])
3. Attention Weights     :
 [[1.   0.   0.  ]
 [0.59 0.41 0.  ]
 [0.31 0.46 0.23]]
4. Context Vectors shape : torch.Size([1, 3, 4])


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# =============================================================================
# TRUTH 1: MULTI-HEAD ATTENTION IS JUST TENSOR RESHAPING
# -----------------------------------------------------------------------------
# We do not create multiple separate linear layers for each head (that would be 
# terribly slow). Instead, we do one massive matrix multiplication, and then 
# magically slice the tensor using .view() and .transpose() to create the heads.
# =============================================================================

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, context_length):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_size = embed_dim // num_heads # e.g., 256 / 8 = 32 dimensions per head
        
        # One massive layer for all heads combined (Much faster for the GPU)
        self.W_query = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_key   = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_value = nn.Linear(embed_dim, embed_dim, bias=False)
        
        # The final linear layer to mix the concatenated heads back together
        self.out_proj = nn.Linear(embed_dim, embed_dim)

        # Register the causal mask so it saves with the model
        self.register_buffer(
            "mask", 
            torch.tril(torch.ones(context_length, context_length))
        )

    def forward(self, x):
        batch_size, num_tokens, embed_dim = x.size() # e.g., [8, 4, 256]
        
        # 1. Get massive Q, K, V matrices: Shape [8, 4, 256]
        Q = self.W_query(x)
        K = self.W_key(x)
        V = self.W_value(x)
        
        # 2. THE SPLIT (The most confusing math in Transformers)
        # We reshape [Batch, Tokens, EmbedDim] -> [Batch, Tokens, Heads, HeadSize]
        # Then we transpose so Heads are grouped together: [Batch, Heads, Tokens, HeadSize]
        # Example: [8, 4, 256] -> [8, 4, 8, 32] -> [8, 8, 4, 32]
        Q = Q.view(batch_size, num_tokens, self.num_heads, self.head_size).transpose(1, 2)
        K = K.view(batch_size, num_tokens, self.num_heads, self.head_size).transpose(1, 2)
        V = V.view(batch_size, num_tokens, self.num_heads, self.head_size).transpose(1, 2)
        
        # 3. Independent Attention Math (Happens for all heads simultaneously!)
        # Q @ K^T
        scores = Q @ K.transpose(-2, -1) / (self.head_size ** 0.5)
        
        # Apply the blindfold (Causal Mask)
        scores = scores.masked_fill(self.mask[:num_tokens, :num_tokens] == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        
        # Multiply by Values
        context = weights @ V # Shape: [8, 8, 4, 32]
        
        # 4. THE CONCATENATION
        # Transpose back to [Batch, Tokens, Heads, HeadSize], then flatten back to [Batch, Tokens, EmbedDim]
        # Example: [8, 8, 4, 32] -> [8, 4, 8, 32] -> [8, 4, 256]
        context = context.transpose(1, 2).contiguous().view(batch_size, num_tokens, embed_dim)
        
        # Mix the newly concatenated heads together
        return self.out_proj(context)


# =============================================================================
# TRUTH 2: THE FEED FORWARD NETWORK (The "Thinking" Layer)
# -----------------------------------------------------------------------------
# Attention acts as a communication protocol between tokens. 
# The Feed Forward Network (FFN) acts as the individual "thinking" phase where 
# each token processes the new context it just received. 
# It expands the data by 4x to learn complex features, then compresses it back.
# =============================================================================

class FeedForward(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim), # Expand representation space
            nn.GELU(),                           # Non-linear activation (Standard in GPT)
            nn.Linear(4 * embed_dim, embed_dim)  # Compress back to normal
        )

    def forward(self, x):
        return self.net(x)


# =============================================================================
# TRUTH 3: THE TRANSFORMER BLOCK (Putting it all together)
# -----------------------------------------------------------------------------
# A full block wraps Attention and FFN with two critical safety features:
# 1. LayerNorm: Keeps the numbers from exploding to infinity.
# 2. Residual Connections (x + function(x)): Allows the original meaning of 
#    the word to bypass the complex math, preventing the network from forgetting.
# =============================================================================

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, context_length):
        super().__init__()
        # Pre-normalization (Standard in GPT-2 and GPT-3)
        self.ln_1 = nn.LayerNorm(embed_dim)
        self.attention = MultiHeadAttention(embed_dim, num_heads, context_length)
        
        self.ln_2 = nn.LayerNorm(embed_dim)
        self.ffn = FeedForward(embed_dim)

    def forward(self, x):
        # 1. Attention Phase with Residual Connection
        # Notice we add the original 'x' back to the output of attention!
        x = x + self.attention(self.ln_1(x))
        
        # 2. Feed Forward Phase with Residual Connection
        x = x + self.ffn(self.ln_2(x))
        
        return x

# =============================================================================
# QUICK VALIDATION SCRIPT
# =============================================================================
if __name__ == "__main__":
    batch_size = 8
    context_length = 4
    embed_dim = 256
    num_heads = 8

    # Create dummy input tensor (simulating output from our data pipeline)
    dummy_input = torch.randn(batch_size, context_length, embed_dim)

    # Initialize the Full Transformer Block
    block = TransformerBlock(embed_dim, num_heads, context_length)

    # Pass the data through the block
    output = block(dummy_input)

    print("--- Transformer Block Architecture Validation ---")
    print("Input Shape :", dummy_input.shape)
    print("Output Shape:", output.shape)
    print("Notice that the dimensions never change! A transformer block takes a")
    print("tensor, infuses it with context and non-linear patterns, and returns")
    print("the exact same shape, ready to be passed into the next block.")

--- Transformer Block Architecture Validation ---
Input Shape : torch.Size([8, 4, 256])
Output Shape: torch.Size([8, 4, 256])
Notice that the dimensions never change! A transformer block takes a
tensor, infuses it with context and non-linear patterns, and returns
the exact same shape, ready to be passed into the next block.
